# MGMT298D: Science and Strategy of AI
## Assignment 1 - Linear Regression with Feature Engineering
### Application: Demand Forecasting at H&M

---

**Instructions:** Complete the exercises below by filling in the `???` placeholders and answering the questions in the designated cells. Run all code cells in order.

## Setup and Data Loading

We'll use H&M sales data to forecast demand for fashion products.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LassoCV, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv("https://raw.githubusercontent.com/ucla-anderson-SSAI/SSAI/refs/heads/main/HMData.csv")

# Available product categories
print("Available products:", df['name'].unique()[:10], "... and more")
print(f"\nTotal rows: {len(df):,}")
print(f"Columns: {list(df.columns[:10])}...")

In [ ]:
# ============================================================
# EXERCISE 1: Choose a product category to analyze
# ============================================================
# Try different products: "Dress", "T-shirt", "Sweater", "Trousers", "Jacket"

SELECTED_PRODUCT = ???  # <-- Fill in a product name (as a string)

df_sub = df[df["name"] == SELECTED_PRODUCT].copy()
print(f"Analyzing {SELECTED_PRODUCT}: {len(df_sub):,} rows")

## Feature Engineering

Feature engineering creates new variables from existing data that help the model make better predictions.

In [ ]:
# Create temporal features
month_cols = ["January", "February", "March", "April", "May", "June",
              "July", "August", "September", "October", "November", "December"]

df_sub["month_num"] = df_sub[month_cols].idxmax(axis=1).apply(lambda x: month_cols.index(x) + 1)

# Lag features: past demand values
for i in range(1, 4):
    df_sub[f"lag_m{i}"] = df_sub.groupby(["id"])["demand"].shift(i)

# Rolling mean: average of last 3 months
df_sub["ma_3"] = (
    df_sub.groupby(["id"], group_keys=False)
    .apply(lambda g: g["demand"].shift(1).rolling(3, min_periods=1).mean(), include_groups=False)
)

# Price change: how much did price change from last period?
df_sub["price_change"] = df_sub.groupby(["id"])["price"].pct_change()

# Fill missing values
df_sub[["lag_m1", "lag_m2", "lag_m3", "ma_3", "price_change"]] = \
    df_sub[["lag_m1", "lag_m2", "lag_m3", "ma_3", "price_change"]].fillna(0)

# Time-based train/test split
months = np.sort(df_sub["month_num"].unique())
train = df_sub[df_sub["month_num"] < months[-1]].copy()
test = df_sub[df_sub["month_num"] == months[-1]].copy()

print(f"Training set: {len(train):,} rows | Test set: {len(test):,} rows")

In [ ]:
# Visualize the engineered features
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Sample a few products for visualization
sample_ids = df_sub['id'].unique()[:3]

# Demand vs lag features
axes[0, 0].scatter(train['lag_m1'], train['demand'], alpha=0.3, s=10)
axes[0, 0].set_xlabel('Last Month Demand (lag_m1)')
axes[0, 0].set_ylabel('Current Demand')
axes[0, 0].set_title('Demand vs. Lag Feature')

# Demand vs price
axes[0, 1].scatter(train['price'], train['demand'], alpha=0.3, s=10, c='orange')
axes[0, 1].set_xlabel('Price')
axes[0, 1].set_ylabel('Current Demand')
axes[0, 1].set_title('Demand vs. Price')

# Demand distribution
axes[1, 0].hist(train['demand'], bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Demand')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Demand')

# Feature correlation heatmap (simplified)
corr_cols = ['demand', 'price', 'lag_m1', 'ma_3']
corr = train[corr_cols].corr()
im = axes[1, 1].imshow(corr, cmap='RdBu', vmin=-1, vmax=1)
axes[1, 1].set_xticks(range(len(corr_cols)))
axes[1, 1].set_yticks(range(len(corr_cols)))
axes[1, 1].set_xticklabels(corr_cols, rotation=45)
axes[1, 1].set_yticklabels(corr_cols)
axes[1, 1].set_title('Feature Correlations')
plt.colorbar(im, ax=axes[1, 1])

plt.tight_layout()
plt.show()

---
## Model Training: Three Progressively Complex Models

We'll build three models with increasing feature complexity:
- **Model A**: Price + Month (baseline)
- **Model B**: Add temporal features (lags, rolling average)
- **Model C**: Add pairwise interactions between features

In [ ]:
# Prepare targets
y_train = train["demand"]
y_test = test["demand"]

# Get actual month columns present in data
actual_month_cols = [col for col in month_cols if col in train.columns]

# ============================================================
# MODEL A: Baseline (price + month)
# ============================================================
modelA_num = ["price"]
train_A = train[modelA_num + actual_month_cols].copy()
test_A = test[modelA_num + actual_month_cols].copy()

scaler_A = StandardScaler()
train_A[modelA_num] = scaler_A.fit_transform(train_A[modelA_num])
test_A[modelA_num] = scaler_A.transform(test_A[modelA_num])

lasso_A = LassoCV(cv=5, random_state=0).fit(train_A, y_train)
pred_A = lasso_A.predict(test_A)
mae_A = mean_absolute_error(y_test, pred_A)

print(f"Model A (Baseline) — MAE = {mae_A:.2f}")

In [ ]:
# ============================================================
# MODEL B: Add temporal features
# ============================================================
modelB_num = ["price", "price_change", "lag_m1", "lag_m2", "lag_m3", "ma_3"]
train_B = train[modelB_num + actual_month_cols].copy()
test_B = test[modelB_num + actual_month_cols].copy()

scaler_B = StandardScaler()
train_B[modelB_num] = scaler_B.fit_transform(train_B[modelB_num])
test_B[modelB_num] = scaler_B.transform(test_B[modelB_num])

lasso_B = LassoCV(cv=5, random_state=0).fit(train_B, y_train)
pred_B = lasso_B.predict(test_B)
mae_B = mean_absolute_error(y_test, pred_B)

print(f"Model B (+ Temporal) — MAE = {mae_B:.2f}")

In [ ]:
# ============================================================
# MODEL C: Add pairwise interactions
# ============================================================
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
poly_tr = poly.fit_transform(train_B[modelB_num])
poly_te = poly.transform(test_B[modelB_num])

X_tr_C = np.hstack([train_B[actual_month_cols].values, poly_tr])
X_te_C = np.hstack([test_B[actual_month_cols].values, poly_te])

lasso_C = LassoCV(cv=5, random_state=0).fit(X_tr_C, y_train)
pred_C = lasso_C.predict(X_te_C)
mae_C = mean_absolute_error(y_test, pred_C)

print(f"Model C (+ Interactions) — MAE = {mae_C:.2f}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MAE comparison
models = ['Model A\n(Baseline)', 'Model B\n(+Temporal)', 'Model C\n(+Interactions)']
maes = [mae_A, mae_B, mae_C]
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']

bars = axes[0].bar(models, maes, color=colors, edgecolor='black')
axes[0].set_ylabel('Mean Absolute Error (MAE)', fontsize=12)
axes[0].set_title('Model Comparison: Lower is Better', fontsize=14)
for bar, mae in zip(bars, maes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                 f'{mae:.1f}', ha='center', fontsize=12, fontweight='bold')

# Actual vs Predicted for best model
axes[1].scatter(y_test, pred_C, alpha=0.3, s=10)
max_val = max(y_test.max(), pred_C.max())
axes[1].plot([0, max_val], [0, max_val], 'r--', label='Perfect Prediction')
axes[1].set_xlabel('Actual Demand', fontsize=12)
axes[1].set_ylabel('Predicted Demand', fontsize=12)
axes[1].set_title('Model C: Actual vs Predicted', fontsize=14)
axes[1].legend()

plt.tight_layout()
plt.show()

---

## Question 1: Interpreting Feature Engineering Impact

Look at the MAE values for the three models above.

**Q1a:** How much did adding temporal features (Model B) improve MAE compared to the baseline (Model A)? Express as a percentage improvement.

*Your answer:*


**Q1b:** Why do you think lag features (yesterday's demand) are so predictive of today's demand? Give a business intuition.

*Your answer:*


---

## Question 2: Experimenting with Regularization

LASSO regression automatically selects the regularization strength (alpha) using cross-validation. Let's explore what happens with different fixed alpha values.

In [ ]:
# ============================================================
# EXERCISE 2: Try different alpha values for Ridge regression
# ============================================================
from sklearn.linear_model import Ridge

# Try these alpha values: 0.01, 0.1, 1.0, 10.0, 100.0
ALPHA = ???  # <-- Fill in an alpha value

ridge = Ridge(alpha=ALPHA)
ridge.fit(train_B, y_train)
pred_ridge = ridge.predict(test_B)
mae_ridge = mean_absolute_error(y_test, pred_ridge)

print(f"Ridge (alpha={ALPHA}) — MAE = {mae_ridge:.2f}")
print(f"Compare to LASSO-CV (Model B) — MAE = {mae_B:.2f}")

In [ ]:
# Visualize effect of regularization
alphas = [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]
ridge_maes = []
num_nonzero_coefs = []

for a in alphas:
    ridge_temp = Ridge(alpha=a).fit(train_B, y_train)
    pred_temp = ridge_temp.predict(test_B)
    ridge_maes.append(mean_absolute_error(y_test, pred_temp))
    num_nonzero_coefs.append(np.sum(np.abs(ridge_temp.coef_) > 0.01))

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogx(alphas, ridge_maes, 'bo-', linewidth=2, markersize=8)
ax.axhline(y=mae_B, color='red', linestyle='--', label=f'LASSO-CV MAE: {mae_B:.2f}')
ax.set_xlabel('Alpha (Regularization Strength)', fontsize=12)
ax.set_ylabel('Test MAE', fontsize=12)
ax.set_title('Effect of Regularization Strength on Prediction Error', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

**Q2:** Based on the plot above, what happens to prediction error (MAE) as alpha gets very large (strong regularization)? Why does this happen?

*Your answer:*


---

## Question 3: Comparing Products

Run the analysis on a different product category and compare results.

In [ ]:
# ============================================================
# EXERCISE 3: Analyze a different product
# ============================================================
PRODUCT_2 = ???  # <-- Choose a different product (e.g., "T-shirt", "Sweater")

# Quick analysis function
def analyze_product(product_name):
    df_p = df[df["name"] == product_name].copy()
    df_p["month_num"] = df_p[month_cols].idxmax(axis=1).apply(lambda x: month_cols.index(x) + 1)
    for i in range(1, 4):
        df_p[f"lag_m{i}"] = df_p.groupby(["id"])["demand"].shift(i)
    df_p["ma_3"] = df_p.groupby(["id"], group_keys=False).apply(
        lambda g: g["demand"].shift(1).rolling(3, min_periods=1).mean(), include_groups=False)
    df_p["price_change"] = df_p.groupby(["id"])["price"].pct_change()
    df_p[["lag_m1", "lag_m2", "lag_m3", "ma_3", "price_change"]] = \
        df_p[["lag_m1", "lag_m2", "lag_m3", "ma_3", "price_change"]].fillna(0)
    
    months = np.sort(df_p["month_num"].unique())
    tr = df_p[df_p["month_num"] < months[-1]]
    te = df_p[df_p["month_num"] == months[-1]]
    
    feats = ["price", "price_change", "lag_m1", "lag_m2", "lag_m3", "ma_3"]
    actual_months = [c for c in month_cols if c in tr.columns]
    X_tr = tr[feats + actual_months].copy()
    X_te = te[feats + actual_months].copy()
    
    sc = StandardScaler()
    X_tr[feats] = sc.fit_transform(X_tr[feats])
    X_te[feats] = sc.transform(X_te[feats])
    
    model = LassoCV(cv=5, random_state=0).fit(X_tr, tr["demand"])
    preds = model.predict(X_te)
    return mean_absolute_error(te["demand"], preds), len(tr), te["demand"].mean()

mae_2, n_rows, avg_demand = analyze_product(PRODUCT_2)
print(f"\n{PRODUCT_2}: MAE = {mae_2:.2f} (avg demand = {avg_demand:.1f}, n = {n_rows:,})")
print(f"{SELECTED_PRODUCT}: MAE = {mae_B:.2f}")

**Q3:** Which product was easier to forecast (lower MAE)? Why might some product categories be more predictable than others?

*Your answer:*


---

## Question 4: Feature Importance

Let's examine which features are most important for prediction.

In [ ]:
# Get feature importance from LASSO coefficients
feature_names = modelB_num + actual_month_cols
coefs = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': lasso_B.coef_
})
coefs['Abs_Coef'] = np.abs(coefs['Coefficient'])
coefs = coefs.sort_values('Abs_Coef', ascending=True)

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
colors = ['green' if c > 0 else 'red' for c in coefs['Coefficient']]
ax.barh(coefs['Feature'], coefs['Coefficient'], color=colors, alpha=0.7)
ax.axvline(x=0, color='black', linewidth=0.5)
ax.set_xlabel('Coefficient Value', fontsize=12)
ax.set_title('Feature Importance (LASSO Coefficients)\nGreen = Positive Effect, Red = Negative Effect', fontsize=14)
plt.tight_layout()
plt.show()

**Q4:** Look at the coefficient for `price`. Is it positive or negative? Does this make business sense? Explain.

*Your answer:*


---

## Question 5: Managerial Application

Imagine you are a data science manager at H&M. You need to decide whether to invest in building more sophisticated demand forecasting models.

**Q5a:** Based on your results, would you recommend using temporal features (lags, rolling averages) in production forecasting models? What's the business case?

*Your answer:*


**Q5b:** What are the potential risks or limitations of relying heavily on lag features (past demand) to predict future demand? When might this approach fail?

*Your answer:*


---

## Question 6: Critical Thinking

**Q6:** If H&M wanted to predict demand for a brand new product that has never been sold before, could they use the lag-based model you built? What alternative approaches might work?

*Your answer:*
